# MCP (Model Context Protocol) サーバー サンプル

Google Colab上でMCPサーバーを構築・テストするサンプルです。

## 構成
1. パッケージのインストール
2. `server.py` の書き出し（ツール・リソース・プロンプト）
3. インプロセステスト（ハンドラを直接呼び出す）
4. サブプロセス起動デモ


## 1. パッケージのインストール


In [ ]:
!pip install mcp httpx nest_asyncio -q
import nest_asyncio
nest_asyncio.apply()
print('インストール完了・イベントループのネスト有効化完了')


## 2. MCPサーバーの実装

以下のツールを実装します:
- `add` : 2つの数値の足し算
- `greet` : 名前を受け取って挨拶を返す
- `get_weather` : ダミーの天気情報を返す


In [ ]:
# server.py を /content/server.py に書き出す
server_lines = [
    "import asyncio\n",
    "from mcp.server import Server\n",
    "from mcp.server.models import InitializationOptions\n",
    "from mcp.server.stdio import stdio_server\n",
    "from mcp import types\n",
    "\n",
    "app = Server('colab-mcp-demo')\n",
    "\n",
    "\n",
    "@app.list_tools()\n",
    "async def list_tools() -> list[types.Tool]:\n",
    "    return [\n",
    "        types.Tool(\n",
    "            name='add',\n",
    "            description='2つの整数を足し算して結果を返します',\n",
    "            inputSchema={\n",
    "                'type': 'object',\n",
    "                'properties': {\n",
    "                    'a': {'type': 'number', 'description': '1つ目の数値'},\n",
    "                    'b': {'type': 'number', 'description': '2つ目の数値'},\n",
    "                },\n",
    "                'required': ['a', 'b'],\n",
    "            },\n",
    "        ),\n",
    "        types.Tool(\n",
    "            name='greet',\n",
    "            description='名前を受け取って日本語で挨拶を返します',\n",
    "            inputSchema={\n",
    "                'type': 'object',\n",
    "                'properties': {\n",
    "                    'name': {'type': 'string', 'description': '挨拶する相手の名前'},\n",
    "                },\n",
    "                'required': ['name'],\n",
    "            },\n",
    "        ),\n",
    "        types.Tool(\n",
    "            name='get_weather',\n",
    "            description='指定した都市のダミー天気情報を返します',\n",
    "            inputSchema={\n",
    "                'type': 'object',\n",
    "                'properties': {\n",
    "                    'city': {'type': 'string', 'description': '都市名 (例: Tokyo)'},\n",
    "                },\n",
    "                'required': ['city'],\n",
    "            },\n",
    "        ),\n",
    "    ]\n",
    "\n",
    "\n",
    "@app.call_tool()\n",
    "async def call_tool(name: str, arguments: dict) -> list[types.TextContent]:\n",
    "    if name == 'add':\n",
    "        result = arguments['a'] + arguments['b']\n",
    "        return [types.TextContent(type='text', text=f\"{arguments['a']} + {arguments['b']} = {result}\")]\n",
    "    elif name == 'greet':\n",
    "        return [types.TextContent(type='text', text=f\"こんにちは、{arguments['name']}さん！MCPサーバーへようこそ\")]\n",
    "    elif name == 'get_weather':\n",
    "        import random\n",
    "        conditions = ['晴れ', '曇り', '雨', '雪']\n",
    "        temp = random.randint(0, 35)\n",
    "        cond = random.choice(conditions)\n",
    "        city = arguments['city']\n",
    "        return [types.TextContent(type='text', text=f\"{city}の天気: {cond}, 気温: {temp}度C (デモデータ)\")]\n",
    "    else:\n",
    "        return [types.TextContent(type='text', text=f'未知のツール: {name}')]\n",
    "\n",
    "\n",
    "@app.list_resources()\n",
    "async def list_resources() -> list[types.Resource]:\n",
    "    return [\n",
    "        types.Resource(\n",
    "            uri='memo://sample',\n",
    "            name='サンプルメモ',\n",
    "            description='デモ用のテキストリソース',\n",
    "            mimeType='text/plain',\n",
    "        )\n",
    "    ]\n",
    "\n",
    "\n",
    "@app.read_resource()\n",
    "async def read_resource(uri: str) -> str:\n",
    "    if uri == 'memo://sample':\n",
    "        return 'これはMCPサーバーのサンプルリソースです。\\nGoogle Colab上で動作しています。'\n",
    "    raise ValueError(f'リソースが見つかりません: {uri}')\n",
    "\n",
    "\n",
    "@app.list_prompts()\n",
    "async def list_prompts() -> list[types.Prompt]:\n",
    "    return [\n",
    "        types.Prompt(\n",
    "            name='summarize',\n",
    "            description='テキストを要約するプロンプト',\n",
    "            arguments=[\n",
    "                types.PromptArgument(name='text', description='要約するテキスト', required=True)\n",
    "            ],\n",
    "        )\n",
    "    ]\n",
    "\n",
    "\n",
    "@app.get_prompt()\n",
    "async def get_prompt(name: str, arguments: dict) -> types.GetPromptResult:\n",
    "    if name == 'summarize':\n",
    "        text = arguments.get('text', '')\n",
    "        return types.GetPromptResult(\n",
    "            description='要約プロンプト',\n",
    "            messages=[\n",
    "                types.PromptMessage(\n",
    "                    role='user',\n",
    "                    content=types.TextContent(\n",
    "                        type='text',\n",
    "                        text=f'次のテキストを3行以内で要約してください:\\n\\n{text}'\n",
    "                    ),\n",
    "                )\n",
    "            ],\n",
    "        )\n",
    "    raise ValueError(f'プロンプトが見つかりません: {name}')\n",
    "\n",
    "\n",
    "async def main():\n",
    "    async with stdio_server() as (read_stream, write_stream):\n",
    "        await app.run(\n",
    "            read_stream,\n",
    "            write_stream,\n",
    "            InitializationOptions(\n",
    "                server_name='colab-mcp-demo',\n",
    "                server_version='1.0.0',\n",
    "                capabilities=app.get_capabilities(\n",
    "                    notification_options=None,\n",
    "                    experimental_capabilities={},\n",
    "                ),\n",
    "            ),\n",
    "        )\n",
    "\n",
    "if __name__ == '__main__':\n",
    "    asyncio.run(main())\n",
]

with open('/content/server.py', 'w', encoding='utf-8') as f:
    f.writelines(server_lines)

import os
assert os.path.exists('/content/server.py'), 'server.py の書き出し失敗'
print('server.py を書き出しました:', os.path.getsize('/content/server.py'), 'bytes')


## 3. サーバーコードの確認


In [ ]:
import os
path = '/content/server.py'
if os.path.exists(path):
    print(f'OK: {path} ({os.path.getsize(path)} bytes)')
else:
    raise FileNotFoundError('server.py が見つかりません。セル2(server.py書き出し)を先に実行してください')


In [ ]:
!cat /content/server.py


## 4. インプロセステスト

Colabではハンドラ関数を直接インポートしてテストするのが簡単です。


In [ ]:
import sys
sys.path.insert(0, '/content')
import server as mcp_server
print('server.py のインポート成功')


In [ ]:
import asyncio

# ツール一覧
async def show_tools():
    tools = await mcp_server.list_tools()
    print('登録されているツール一覧:')
    for tool in tools:
        print(f'  {tool.name}: {tool.description}')

asyncio.run(show_tools())


In [ ]:
import asyncio

# add ツールのテスト
async def test_add():
    result = await mcp_server.call_tool('add', {'a': 42, 'b': 58})
    print('add ツール結果:', result[0].text)

asyncio.run(test_add())


In [ ]:
import asyncio

# greet ツールのテスト
async def test_greet():
    result = await mcp_server.call_tool('greet', {'name': 'Claude'})
    print('greet ツール結果:', result[0].text)

asyncio.run(test_greet())


In [ ]:
import asyncio

# get_weather ツールのテスト
async def test_weather():
    result = await mcp_server.call_tool('get_weather', {'city': 'Tokyo'})
    print('get_weather ツール結果:', result[0].text)

asyncio.run(test_weather())


In [ ]:
import asyncio

# リソースのテスト
async def test_resources():
    resources = await mcp_server.list_resources()
    print('リソース一覧:')
    for r in resources:
        print(f'  {r.name} ({r.uri})')
    content = await mcp_server.read_resource('memo://sample')
    print('\nリソース内容:\n', content)

asyncio.run(test_resources())


In [ ]:
import asyncio

# プロンプトのテスト
async def test_prompts():
    prompts = await mcp_server.list_prompts()
    print('プロンプト一覧:')
    for p in prompts:
        print(f'  {p.name}: {p.description}')
    prompt_result = await mcp_server.get_prompt(
        'summarize', {'text': 'MCPはAnthropicが開発したオープンプロトコルです。'}
    )
    print('\n生成されたプロンプト:')
    print(prompt_result.messages[0].content.text)

asyncio.run(test_prompts())


## 5. サブプロセスとしてMCPサーバーを起動する（上級者向け）

実際のMCPクライアント（Claude Desktopなど）と接続する場合は、
stdioトランスポートでサーバーをプロセス起動します。

> 以下はデモとして短時間起動してすぐ終了するサンプルです。


In [ ]:
import subprocess, time

proc = subprocess.Popen(
    [sys.executable, '/content/server.py'],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print(f'MCPサーバー起動 (PID: {proc.pid})')
time.sleep(1)

if proc.poll() is None:
    print('サーバーは正常に動作中です')
else:
    print('サーバーが終了しました:', proc.stderr.read().decode())

proc.terminate()
print('サーバーを停止しました')


## カスタマイズのヒント

| やりたいこと | 変更箇所 |
|---|---|
| ツールを追加する | `list_tools()` に Tool を追加 + `call_tool()` に分岐追加 |
| 外部APIを叩く | `call_tool()` 内で `httpx.AsyncClient` を使用 |
| DBに接続する | `read_resource()` 内で SQLAlchemy など利用 |
| SSEトランスポートに変更 | `mcp.server.sse` モジュールを利用 |

## 参考リンク
- [MCP公式ドキュメント](https://modelcontextprotocol.io)
- [mcp Python SDK (GitHub)](https://github.com/modelcontextprotocol/python-sdk)
